# Fitting Methods

Here we will explore the various fitting methods in AstroPhot. You have already encountered some of the methods, but here we will take a more systematic approach and discuss their strengths/weaknesses. Each method will be applied to the same problem with the same initial conditions so you can see how they operate.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

import astrophot as ap

## The test problem

Every fitter below gets the same job: recover the parameters of a single Sersic galaxy and the flat sky it sits on from a 50 x 50 pixel mock image. You can skip the code in the next cell to start, it makes the mock image and leaves us with a few things that are used throughout:

- `true_params` are the values used to make the image and `init_params` are the (deliberately wrong) values that every fit starts from. Feel free to play around with these to see what happens!
- `make_model()` returns a new model at the starting values, a group model made of the sky and the Sersic.
- `show_fit(model)` plots the model and its residuals, at the starting values and after fitting.
- `param_names` and `true_values` are the names and true values of the parameters in the order the fitters use, for labelling plots.

In [ ]:
# The values used to make the mock image, and the values every fit starts from
true_params = {
    "sky": dict(I0=25.0),
    "sersic": dict(center=[24.3, 26.1], q=0.6, PA=60 * np.pi / 180, n=2.0, Re=8.0, Ie=100.0),
}
init_params = {
    "sky": dict(I0=20.0),
    "sersic": dict(center=[22.0, 28.0], q=0.8, PA=30 * np.pi / 180, n=1.2, Re=12.0, Ie=50.0),
}

target = ap.TargetImage(data=np.zeros((50, 50)), pixelscale=1.0)


def make_model(params=init_params):
    # A new sky + Sersic model, at the starting values unless other ones are given.
    # Every parameter has a value, so there is nothing that needs model.initialize()
    sky = ap.Model(name="sky", model_type="flat sky model", target=target, **params["sky"])
    sersic = ap.Model(
        name="sersic", model_type="sersic galaxy model", target=target, **params["sersic"]
    )
    return ap.Model(name="model", model_type="group model", models=[sky, sersic], target=target)


# The mock image is the true model with the noise of counting its photons, where
# the sky sets the level of the noise away from the galaxy
truth = make_model(true_params)
image = truth().data.detach().cpu().numpy()
target.data = image + np.random.default_rng(42).normal(scale=np.sqrt(image))
target.variance = image

param_names = truth.build_params_array_names()
true_values = truth.get_values().detach().cpu().numpy()


def show_fit(model):
    # Model and residuals at the starting values, then the same for the model as it is now
    fig, axarr = plt.subplots(1, 4, figsize=(24, 5))
    plt.subplots_adjust(wspace=0.1)
    for axes, stage, when in ((axarr[:2], make_model(), "before"), (axarr[2:], model, "after")):
        ap.plots.model_image(fig, axes[0], stage)
        axes[0].set_title(f"Model {when} optimization")
        ap.plots.residual_image(fig, axes[1], stage, normalize_residuals=True)
        axes[1].set_title(f"Residuals {when} optimization")
    plt.show()


fig, ax = plt.subplots(figsize=(6, 6))
ap.plots.target_image(fig, ax, target)
ax.axis("off")
plt.show()

## Common fitter options

Every fitter is used the same way. You create it from a model along with any options, then call `fit`:

```python
res = ap.fit.LM(model, max_iter=50).fit()
```

The fit changes the model itself, when it is done the parameters of `model` hold the result. `fit` returns the fitter, so `res` above is a record of what happened: `res.message` says why the fit stopped, and `ap.fit.LM`, `ap.fit.Iter`, `ap.fit.IterParam` and `ap.fit.Slalom` also keep the loss at every iteration in `res.loss_history` and the parameter values that went with it in `res.lambda_history`. The samplers store their samples in `res.chain`.

The options in the table below are shared between the fitters. The first column is the name of the keyword argument, the second is what it controls, and the third lists the fitters on which it has an effect. Options that belong to a single fitter are listed in the section for that fitter.

| Option | What it does | Used by |
| --- | --- | --- |
| `initial_state` | Parameter values to start from, as a 1D array in the order of `model.get_values()`. By default the fit starts from the values currently in the model. | All. `MALA` and `MHMCMC` also take a 2D array with one row per chain |
| `max_iter` | For the optimizers, the number of iterations after which the fit gives up, the message then starts with "fail". For the samplers, the number of samples to draw per chain. The default is 100 for `LM`, `Iter` and `IterParam`, 100 per parameter for `ScipyFit` and 1000 for `Slalom` and the samplers. | All |
| `relative_tolerance` | The fit has converged once the fractional improvement of the loss from one iteration to the next falls below this. The exact rule differs a little between fitters and is given in each section. The default is 1e-5 for `LM` and `IterParam`, 1e-4 for `Slalom` and 1e-3 for `Iter`. | `LM`, `Iter`, `IterParam`, `Slalom` |
| `likelihood` | `"gaussian"` (the default) is a $\chi^2$ fit using the variance of the target image. `"poisson"` treats the pixel values as counts, see the Poisson noise model tutorial. | All except `Iter`, which is always a $\chi^2$ fit and does not accept this option |
| `fit_valid` | With `True` (the default) parameters can never leave their valid range, such as $0 < q < 1$. This is done by fitting in a transformed space where every value is allowed. With `False` the fit works on the plain values and nothing stops a step from leaving the range. | `LM`, `IterParam`. `Slalom` always fits in the transformed space, `ScipyFit` gives the limits to scipy as bounds, and the samplers work on the plain values |
| `save_steps` | A file name at which to save the state of the model after every iteration, in the HDF5 format that `model.load_state` reads. Put `{step}` in the name to get one file per iteration, as in `"fit_step_{step}.hdf5"`, without it the file is overwritten and always holds the latest iteration. Not saved by default. | `LM`, `Iter`, `IterParam`, `Slalom` |
| `verbose` | How much the fitter reports as it runs. By default this follows the AstroPhot logger: progress is logged at the INFO level and extra detail at DEBUG, so `ap.set_logger(level="WARNING")` quiets every fitter at once and `level="DEBUG"` shows the most (see "Tracking output" in the getting started tutorial). Give a number to choose for one fitter instead: 0 for no progress, 1 for progress, 2 to add the extra detail. | `LM`, `Iter`, `IterParam`, `ScipyFit`, `Slalom`. Of the samplers, `MALA` and `HMC` show a progress bar instead, which `progress_bar=False` turns off |

There are two more fitters that are not covered here because they need a different setup: `ap.fit.BatchLM` fits the same model to many images at once (see the image types tutorial), and `ap.fit.LMConstraint` adds priors and constraints to an `LM` fit (see the fitting with priors tutorial).

## Levenberg-Marquardt

This fitter is identified as `ap.fit.LM` and it employs a variant of the second order Newton's method to converge very quickly to the local minimum. This is the generally accepted best algorithm for most use cases in $\chi^2$ minimization. If you don't know what to pick, start with this minimizer. The LM optimizer bridges the gap between first-order gradient descent and second order Newton's method. When far from the minimum, Newton's method is unstable and can give wildly wrong results, so LM takes gradient descent steps. However, near the minimum it switches to the Newton's method which has "quadratic convergence" this means that it takes only a few iterations to converge to several decimal places. This can be represented as:

$(H + LI)h = g$

Where H is the Hessian matrix of second derivatives, L is the damping parameter, I is the identity matrix, h is the step we will take in parameter space, and g is the gradient. We solve this linear system for h to get the next update step. The "L" scale parameter goes from L >> 1 which represents gradient descent to L << 1 which is Newton's Method. When L >> 1 the hessian is effectively zero and we get $h = g/L$ which is just gradient descent with $1/L$ as the learning rate.

In AstroPhot the details differ, but the concept is the same. The Hessian is approximated as $J^TWJ$, where $J$ is the Jacobian of the model image with respect to the parameters and $W$ is the weight (inverse variance) of each pixel, so no second derivatives are needed. The damping rescales the Hessian rather than adding the identity to it: the diagonal is multiplied by $1 + L$ and the off-diagonal elements are divided by $1 + L$, which keeps the step independent of the units of each parameter. After a step that improves the fit L is divided by `Ldn`. A step that makes the fit worse, or that improves it very differently from what the linear approximation predicted, is thrown away and tried again with L multiplied by `Lup`.

The fit ends with the message "success" when the $\chi^2$ improved by a fraction less than `relative_tolerance` in the last step while L is below 0.1, meaning Newton steps are being taken and they no longer help. It ends with "success by immobility" when no step that improves the fit can be found, or when the last 10 steps achieved less than `relative_tolerance` between them. In that case it is worth checking the result.

LM can handle a lot of scenarios and converge to the minimum. Keep in mind, however, that it is seeking a local minimum, so it is best to start off the algorithm as close as possible to the best fit parameters. AstroPhot can automatically initialize, as discussed in other notebooks, but even that needs help sometimes (often in the form of a segmentation map).

The main drawback of LM is its memory consumption which goes as $\mathcal{O}(PN)$ where P is the number of pixels and N is the number of parameters.

Options specific to `ap.fit.LM`:

- `L0`: the damping parameter L to start with. Default 1.
- `Lup`, `Ldn`: the factors by which L is increased after a rejected step and decreased after a good one. Defaults 11 and 9.
- `max_step_iter`: the most values of L to try in a single iteration, each of which costs an evaluation of the model. Default 10.
- `ndf`: the degrees of freedom used for the reported $\chi^2$/DoF. Default is the number of unmasked pixels minus the number of parameters.
- `W`: weights to use in place of those of the target image, as an array like `target.weight`.
- `constraint`: an `ap.fit.LMConstraint` to add priors or constraints to the fit, see the fitting with priors tutorial.
- `forward`, `jacobian`: functions to use in place of the model for computing the image and its Jacobian from the vector of parameters. These are for advanced uses such as fitting a function of the model.
- `fit(update_uncertainty=True)`: after fitting, the uncertainty of every parameter is set from the covariance matrix (see below). Pass `False` to `fit` to skip this.

In [ ]:
model = make_model()

res_lm = ap.fit.LM(model).fit()
print(res_lm.message)

In [ ]:
show_fit(model)

Now that LM has found the $\chi^2$ minimum, we can do a really neat trick. Since LM needs the hessian matrix, we have access to the hessian matrix at the minimum. This is in fact equal to the Fisher information matrix. If we take the matrix inverse of this matrix then we get the covariance matrix for a multivariate gaussian approximation of the likelihood near the minimum, which is available as `res_lm.covariance_matrix`. With the covariance matrix we can create a corner plot just like we would with an MCMC. The ellipses below are the 1 and 2 sigma contours of that gaussian and the straight lines are the true values. We will see later that the MCMC methods (at least the ones which converge) produce very similar results!

In [ ]:
fig, ax = ap.plots.covariance_matrix(
    res_lm.covariance_matrix.detach().cpu().numpy(),
    model.get_values().detach().cpu().numpy(),
    labels=param_names,
    reference_values=true_values,
)

## Iterative Fit (models)

This iterative fitter is identified as `ap.fit.Iter`, this method is generally employed for large models where it is not feasible to hold all the relevant data in memory at once. The iterative fitter will cycle through the models in a `GroupModel` object and fit them one at a time to the image with `ap.fit.LM`, using the residuals from the previous cycle. This can be a very robust way to deal with some fits, especially if the overlap between models is not too strong. It is however more dependent on good initialization than other methods like the Levenberg-Marquardt. Also, it is possible for the Iter method to get stuck in a local minimum under certain circumstances.

Note that while the Iterative fitter needs a `GroupModel` object to iterate over, it is not necessarily true that the sub models are `ComponentModel` objects, they could be `GroupModel` objects as well. In this way it is possible to cycle through and fit "clusters" of objects that are nearby, so long as it doesn't consume too much memory.

By only fitting one model at a time it is possible to get caught in a local minimum, or to get out of a local minimum that a different fitter was stuck in. For this reason it can be good to mix-and-match the iterative optimizers so they can help each other get unstuck if a fit is very challenging.

Our model is a group of two, so here `Iter` alternates between fitting the sky and fitting the Sersic. A sky overlaps with the galaxy in every pixel, which makes this a hard case for `Iter`: watch how many more cycles it needs than LM needed iterations.

The fit ends with "success" once the $\chi^2$ changes by a fraction less than a tenth of `relative_tolerance` for two cycles in a row. `Iter` does not compute a covariance matrix or parameter uncertainties, follow it with an `LM` fit if you need them and it fits in memory.

Options specific to `ap.fit.Iter`:

- `lm_kwargs`: a dictionary of options for the `ap.fit.LM` that fits each sub model. These are added to the defaults `{"verbose": 0, "relative_tolerance": 1e-3, "max_iter": 15}`, so the sub fits report nothing and stop early unless you say otherwise. It is not worthwhile to polish one model when its neighbours are about to move.

In [ ]:
model = make_model()

res_iter = ap.fit.Iter(model).fit()

In [ ]:
print(f"{res_iter.message} after {res_iter.iteration} cycles")
show_fit(model)

## Iterative Fit (Param)

This iterative fitter is identified as `ap.fit.IterParam`, this is generally employed for large and interconnected models where it is not feasible to hold all the relevant data in memory at once. Unlike `ap.fit.Iter` which is intended to cycle through the sub models in a group model, this fitter iterates through parameters. The set of parameters which make up the model is broken into chunks and then fitting proceeds only on those chunks, rather than on all parameters simultaneously. For large models that have lots of interconnected/shared parameters, it doesn't really make sense to cycle through one sub-model at a time as optimizing that model may throw another model that is sharing a parameter into a bad part of parameter space. Thus `ap.fit.IterParam` is safe to use on any AstroPhot model without concern for this issue, the fitter will industriously proceed to high likelihood solutions monotonically.

The tradeoff for this fitter is the same as for the other iterative fitter, if there are strong covariances in the likelihood structure then this fitter can take a long time to converge. The advantage here is that as the user you may take greater control over the combinations if you wish, which is what we do below. Each chunk is fit with a Levenberg-Marquardt step, and the fit ends by the same rules as `ap.fit.LM`.

Options specific to `ap.fit.IterParam`:

- `chunks`: how to split up the parameters. With an integer like `6`, the parameters are fit `6` at a time in the order of `model.get_values()` (the last chunk may be smaller). Alternatively give a tuple with one selector per chunk, each one a list or array as long as the number of parameters with 1 (or `True`) for the parameters in that chunk. For our 8 parameter model, `([0,1,1,0,0,0,0,0], [0,0,0,1,1,0,0,0], [1,0,0,0,0,1,1,1])` makes three chunks to fit the `x, y` then `q, PA` then the sky together with `n, Re, Ie`. Note that you do not need to make the chunks exclusive, it is totally fine to have a parameter pop up in multiple chunks! Default 50.
- `chunk_order`: with `"sequential"` (the default) the chunks are fit in the order given, with `"random"` a new random order is decided each iteration.
- `L0`, `Lup`, `Ldn`, `max_step_iter`, `ndf`, `W`: the same as for `ap.fit.LM`.
- `fit(update_uncertainty=True)`: the same as for `ap.fit.LM`.

In [ ]:
model = make_model()

# One selector per chunk, in the order of param_names. The sky goes with the
# parameters of the profile since those are the ones it can be traded off against
chunks = (
    [0, 1, 1, 0, 0, 0, 0, 0],  # center
    [0, 0, 0, 1, 1, 0, 0, 0],  # q, PA
    [1, 0, 0, 0, 0, 1, 1, 1],  # sky, n, Re, Ie
)
res_iterparam = ap.fit.IterParam(model, chunks=chunks).fit()

In [ ]:
show_fit(model)

The `ap.fit.IterParam` fitter can also generate a covariance matrix of uncertainties, just keep in mind that it only evaluates the covariances for parameters in the same chunk. Below you can see this as circles for every pair of parameters that were in different chunks. Each chunk was also evaluated with the other parameters held fixed, so the uncertainties are smaller than the ones from the full `ap.fit.LM` covariance matrix above.

In [ ]:
fig, ax = ap.plots.covariance_matrix(
    res_iterparam.covariance_matrix.detach().cpu().numpy(),
    model.get_values().detach().cpu().numpy(),
    labels=param_names,
    reference_values=true_values,
)

## Scipy Minimize

Any AstroPhot model becomes a function `model(x)` where `x` is a 1D tensor of
all the current dynamic parameters and the output is the model image. Similarly,
any model can be called like ``model.gaussian_log_likelihood(x)`` to get the log
likelihood for optimization purposes. This functional format is common for
external packages to use. AstroPhot includes a wrapper, `ap.fit.ScipyFit`, to
access the `scipy.optimize.minimize` minimizer list. It gives scipy the negative
log likelihood, its gradient from automatic differentiation, and the valid range
of each parameter as bounds. Only the scipy methods which accept bounds are
available.

Typically, the AstroPhot LM optimizer is faster and more accurate than the Scipy
ones. The Scipy minimizers are intended for very general use and only see a
single number to minimize, while the LM optimizer is specifically optimized for
gaussian log likelihoods and makes use of the residual in every pixel.

The standard method that AstroPhot will pick for Scipy minimize is ``L-BFGS-B``.
Like LM it uses curvature to take better steps than gradient descent, but it
builds its estimate of the Hessian from the gradients at the previous steps
rather than computing one. On a problem as simple as this one it arrives at the
same minimum as LM, it just takes more iterations to get there. That is not
guaranteed on harder problems, where a minimizer can report that it terminated
successfully when in fact it hasn't converged. Consider this a lesson in
trusting the "success" message from any optimizer. It turns out to be very
challenging to identify if an optimizer is at a minimum, let alone the global
minimum.

Scipy decides for itself when to stop, so `relative_tolerance` is not used
here. `max_iter` is given to scipy as the limit on its iterations, except for
the `"TNC"` method where it limits the evaluations of the likelihood. The full report from scipy is kept as `res.scipy_res`.

Options specific to `ap.fit.ScipyFit`:

- `method`: the scipy minimizer to use, one of `"L-BFGS-B"` (the default),
  `"Nelder-Mead"`, `"TNC"`, `"SLSQP"`, `"Powell"` or `"trust-constr"`.
- `ndf`: the degrees of freedom used for the reported loss. Default is the
  number of unmasked pixels minus the number of parameters.

In [ ]:
model = make_model()

res_scipy = ap.fit.ScipyFit(model).fit()
print(res_scipy.scipy_res)

In [ ]:
show_fit(model)

## Gradient Descent (Slalom)

A gradient descent fitter uses local gradient information to determine the direction of increased likelihood in parameter space. The challenge with gradient descent is choosing a step size. The `Slalom` algorithm developed for AstroPhot, identified as `ap.fit.Slalom`, uses a few samples along the gradient direction to determine a parabola which it can then jump to the minimum of. In some sense this is like a 1D version of the Levenberg-Marquardt algorithm and the 1 dimension it choses is that along the gradient (plus momentum).

The fit ends with "success" when 5 steps in a row have together improved the negative log likelihood by a fraction less than `relative_tolerance`. This is a heuristic, not a guarantee. Like other gradient descent algorithms Slalom slows down a lot near the minimum, so the default tolerance is set to stop once most of the improvement has been made.

As you see below, `Slalom` ends with a decent fit, though not as good as some other methods (Levenberg-Marquardt). This is typically the case. The residuals look fine by eye, but the $\chi^2$/DoF is about 1.09 where LM reached 1.01, and the sky, `n`, `Re` and `Ie` are still some way from their true values. Those four can be traded off against each other at little cost in likelihood, and a long narrow valley like that is where gradient descent is slowest. However, gradient descent can be very helpful for complex optimization tasks, because it is a slower optimization algorithm, it can be more stable in some circumstances. Try using it in cases where LM fails to get things back on track. Just make sure to finish off with an LM round to ensure you have settled into the minimum.

Options specific to `ap.fit.Slalom`:

- `S`: the size of the first step in parameter space. After that the step size is found by the algorithm, so this rarely needs to change. Default 1e-4.
- `momentum`: how much of the direction of the previous steps is kept, between 0 (plain gradient descent) and just under 1. Default 0.9.
- `report_freq`: progress is reported once every this many steps. Default 10.

In [ ]:
model = make_model()

res_grad = ap.fit.Slalom(model).fit()

In [ ]:
show_fit(model)

## Metropolis Adjusted Langevin Algorithm (MALA)

This is one of the simplest gradient based samplers, and is very powerful. The standard Metropolis Hastings algorithm will use a gaussian proposal distribution then use the Metropolis Hastings accept/reject stage. MALA uses gradient information to determine a better proposal distribution locally (while maintaining detailed balance) and then uses the Metropolis Hastings accept/reject stage. The `ap.fit.MALA` fitter object is just a basic wrapper over the `ap.fit.func.mala` function, so feel free to check it out if you want more details on this simple and powerful sampler!

Samplers work best when they know the shape of the posterior in advance, and we already have a good estimate of it: the covariance matrix from LM. So below we first run LM, which also means the chains start at a high likelihood location and no burn-in is needed, then pass its covariance matrix as the mass matrix.

The samples are stored in `res.chain` with shape (`max_iter`, `chains`, number of parameters) and their log likelihoods in `res.logp`. The model is left at the sample with the highest likelihood.

Options specific to `ap.fit.MALA`:

- `chains`: the number of chains to run, they are evaluated together so several cost little more than one. Default 4. To start every chain somewhere different give a 2D `initial_state` with one row per chain, the number of rows then sets the number of chains.
- `epsilon`: the step size, in units set by the mass matrix. With a mass matrix that matches the posterior a value near 1 works well, without one it needs to be about as small as the best constrained parameter. The acceptance rate is shown in the progress bar, lower `epsilon` if it is far below a half. Default 1e-2.
- `mass_matrix`: a matrix with the shape of the covariance matrix, used to shape the proposals. Default is the identity matrix.
- `progress_bar`: show a progress bar while sampling. Default `True`.

In [ ]:
model = make_model()

# Use LM to start the sampler at a high likelihood location, no burn-in needed!
res_lm = ap.fit.LM(model).fit()

res_mala = ap.fit.MALA(
    model,
    chains=4,
    max_iter=300,
    epsilon=1.0,
    mass_matrix=res_lm.covariance_matrix.detach().cpu().numpy(),
).fit()

In [ ]:
# corner plot of the posterior
fig, ax = ap.plots.corner_plot(res_mala.chain, labels=param_names, reference_values=true_values)

## Hamiltonian Monte-Carlo (HMC)

The `ap.fit.HMC` takes a fixed number of steps at a fixed step size following Hamiltonian dynamics. This is in contrast to NUTS which attempts to optimally choose these parameters. The simplest way to think of HMC is as performing a number of MALA steps all in one go, so if `leapfrog_steps = 10` then HMC is very similar to running MALA then taking every tenth step and adding it to the chain. HMC results will still have autocorrelation which will depend on the problem and choice of step parameters.

This fitter is a wrapper for the HMC sampler of the `pyro` package, so `pyro-ppl` must be installed and it is only available with the PyTorch backend. It runs a single chain, which is stored in `res.chain` as a tensor with shape (`max_iter`, number of parameters). The model is left at the last sample.

Options specific to `ap.fit.HMC`:

- `epsilon`: the size of each leapfrog step, in units set by the mass matrix. Default 1e-4.
- `leapfrog_steps`: the number of leapfrog steps taken to produce each sample. Default 10.
- `inv_mass`: the inverse mass matrix, for which a covariance matrix of the parameters is the ideal choice. By default it starts as the identity matrix and is adapted during warmup.
- `warmup`: the number of steps taken, and thrown away, before sampling starts. Default 100. Below we start from the LM solution with its covariance matrix, so there is nothing to warm up.
- `prior`: a `pyro` distribution to use as the prior on the vector of parameters. Default is a very wide gaussian centred on the starting values.
- `progress_bar`: show a progress bar while sampling. Default `True`.
- `hmc_kwargs`, `mcmc_kwargs`: dictionaries of further options for the `HMC` and `MCMC` objects of `pyro`.

In [ ]:
model = make_model()

# Use LM to start the sampler at a high likelihood location, no burn-in needed!
res_lm = ap.fit.LM(model).fit()

# Run the HMC sampler
res_hmc = ap.fit.HMC(
    model,
    warmup=1,
    max_iter=150,
    epsilon=0.2,
    leapfrog_steps=10,
    inv_mass=res_lm.covariance_matrix,
).fit()

In [ ]:
# corner plot of the posterior
fig, ax = ap.plots.corner_plot(
    res_hmc.chain.detach().cpu().numpy(), labels=param_names, reference_values=true_values
)

## Metropolis Hastings

This is the more standard kind of MCMC, which needs no gradients, identified with `ap.fit.MHMCMC`. Under the hood, this is just a wrapper for the excellent `emcee` package (which must be installed), if you want to take advantage of more `emcee` features you can very easily use `ap.fit.MHMCMC` as a starting point. `emcee` runs a group of walkers, each one proposes a move using the position of another walker and then accepts or rejects it with a Metropolis Hastings step. However, one should keep in mind that for large models it can take exceedingly long to actually converge to the posterior. Instead of waiting that long, we demonstrate the functionality with 200 steps of 16 walkers, but suggest using MALA for any real world problem. Still, if there is something MALA can't handle (a function that isn't differentiable) then MHMCMC can save the day (even if it takes all day to do it).

By default there are two walkers per parameter, which is the fewest that `emcee` allows, and they start scattered by 1% around the current parameter values. The samples of all the walkers are stored together in `res.chain`, with shape (`max_iter` x walkers, number of parameters). The model is left at the last sample.

`ap.fit.MHMCMC` has no options of its own beyond the common ones, but there are several for its `fit` method:

- `state`: where to start the walkers, as a 2D array with one row per walker. This is how to use more walkers than the default.
- `nsamples`: the number of steps to take, in place of `max_iter`.
- `restart_chain`: with `False` the new samples are added to the end of `res.chain` from an earlier call to `fit`, rather than replacing it. Default `True`.
- `flat_chain`: with `False` the chain keeps a separate axis for the walkers, with shape (steps, walkers, number of parameters). Default `True`.

In [ ]:
model = make_model()

# Use LM to start the sampler at a high likelihood location
res_lm = ap.fit.LM(model).fit()

# Run the MHMCMC sampler
res_mh = ap.fit.MHMCMC(model, max_iter=200).fit()

In [ ]:
# corner plot of the posterior
# note that, even 3200 samples is not enough to overcome the autocorrelation so the posterior has not converged.
fig, ax = ap.plots.corner_plot(res_mh.chain, labels=param_names, reference_values=true_values)